# 02 — Pipeline d'ingestion
**Phase 2 / Semaine 2**

Objectifs :
1. Vérifier la connexion à PostGIS
2. Charger le GTFS statique (routes + stops) dans la DB
3. Activer l'insertion temps réel des positions dans `vehicle_positions`
4. Tester une collecte manuelle et vérifier les données en base

**Prérequis :**
- `docker compose up db -d` lancé
- Fichiers GTFS statiques dans `data/gtfs_static/`

In [3]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

print('Environnement chargé ✓')

Environnement chargé ✓


## 1. Vérifier la connexion PostGIS

In [4]:
from src.utils.db import check_connection, engine
from sqlalchemy import text

assert check_connection(), "❌ Impossible de se connecter — vérifiez que docker compose up db -d est lancé"
print('Connexion PostGIS OK ✓')

# Vérifier que PostGIS est bien activé
with engine.connect() as conn:
    version = conn.execute(text("SELECT PostGIS_Version()")).scalar()
    print(f'PostGIS version : {version}')

Connexion PostGIS OK ✓
PostGIS version : 3.4 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


## 2. Charger le GTFS statique

On charge `routes.txt` et `stops.txt` dans les tables PostGIS définies dans `docker/init.sql`.

> **Note :** `stop_times.txt` fait ~1M de lignes — le chargement peut prendre 1-2 minutes.

In [6]:
import pandas as pd
from pathlib import Path

GTFS_DIR = Path('../data/gtfs_static')

# Vérifier que les fichiers existent
required = ['routes.txt', 'stops.txt', 'trips.txt', 'stop_times.txt']
for f in required:
    path = GTFS_DIR / f
    status = '✓' if path.exists() else '❌ MANQUANT'
    print(f'{status}  {f}')

✓  routes.txt
✓  stops.txt
✓  trips.txt
✓  stop_times.txt


In [8]:
# ── Charger routes ────────────────────────────────────────────────────────────
routes_df = pd.read_csv(GTFS_DIR / 'routes.txt', dtype=str)
print(f'Routes : {len(routes_df)} lignes')
print(routes_df[['route_id', 'route_short_name', 'route_long_name', 'route_type']].head())

Routes : 216 lignes
  route_id route_short_name   route_long_name route_type
0        1                1   Ligne 1 - Verte          1
1        2                2  Ligne 2 - Orange          1
2        4                4   Ligne 4 - Jaune          1
3        5                5   Ligne 5 - Bleue          1
4       10               10       De Lorimier          3


In [9]:
from sqlalchemy import text
from src.utils.db import engine

# Insérer routes dans PostGIS
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE routes CASCADE"))
    for _, row in routes_df.iterrows():
        conn.execute(text("""
            INSERT INTO routes (route_id, route_short_name, route_long_name, route_type)
            VALUES (:id, :short, :long, :type)
            ON CONFLICT (route_id) DO NOTHING
        """), {
            'id':    row.get('route_id'),
            'short': row.get('route_short_name'),
            'long':  row.get('route_long_name'),
            'type':  int(row['route_type']) if pd.notna(row.get('route_type')) else None,
        })

print(f'✓ {len(routes_df)} routes insérées')

✓ 216 routes insérées


In [10]:
# ── Charger stops avec géométrie PostGIS ─────────────────────────────────────
stops_df = pd.read_csv(GTFS_DIR / 'stops.txt', dtype=str)
print(f'Arrêts : {len(stops_df)} lignes')
print(stops_df[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']].head())

Arrêts : 8951 lignes
        stop_id          stop_name   stop_lat    stop_lon
0  STATION_M118  STATION ANGRIGNON  45.446397  -73.603293
1            43  Station Angrignon  45.446466  -73.603118
2         43-01  Station Angrignon  45.446319  -73.603835
3  STATION_M120       STATION MONK  45.451166  -73.593265
4            42       Station Monk  45.451158  -73.593242


In [11]:
import time

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE stops CASCADE"))
    
batch = [
    {
        'id':   row['stop_id'],
        'name': row.get('stop_name'),
        'lat':  float(row['stop_lat']),
        'lon':  float(row['stop_lon']),
    }
    for _, row in stops_df.iterrows()
]

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE stops CASCADE"))
    conn.execute(
        text("""
            INSERT INTO stops (stop_id, stop_name, location)
            VALUES (:id, :name, ST_SetSRID(ST_MakePoint(:lon, :lat), 4326))
            ON CONFLICT (stop_id) DO NOTHING
        """),
        batch
    )

print(f'✓ {len(stops_df)} arrêts insérés avec géométrie PostGIS')


✓ 8951 arrêts insérés avec géométrie PostGIS


## 4. Prochaines étapes

- [ ] Télécharger le GTFS statique STM (https://www.stm.info/fr/a-propos/developpeurs)
- [ ] Charger `stops.txt` et `routes.txt` dans PostGIS
- [ ] Démarrer `docker-compose up db` et vérifier la connexion avec `src/utils/db.py`
- [ ] Passer au notebook `02_pipeline_ingestion.ipynb`

In [12]:
# Vérification rapide
with engine.connect() as conn:
    n_routes = conn.execute(text("SELECT COUNT(*) FROM routes")).scalar()
    n_stops  = conn.execute(text("SELECT COUNT(*) FROM stops")).scalar()
    sample   = conn.execute(text("""
        SELECT stop_id, stop_name,
               ST_X(location) AS lon, ST_Y(location) AS lat
        FROM stops LIMIT 3
    """)).fetchall()

print(f'Routes en DB : {n_routes}')
print(f'Arrêts en DB : {n_stops}')
print('\nExemple arrêts :')
for row in sample:
    print(f'  {row.stop_id} | {row.stop_name} | ({row.lat:.4f}, {row.lon:.4f})')

Routes en DB : 216
Arrêts en DB : 8951

Exemple arrêts :
  STATION_M118 | STATION ANGRIGNON | (45.4464, -73.6033)
  43 | Station Angrignon | (45.4465, -73.6031)
  43-01 | Station Angrignon | (45.4463, -73.6038)


## 3. Collecter et insérer les positions en temps réel

On appelle directement les fonctions du collecteur et on insère en DB.

In [13]:
from src.collector.gtfs_client import GTFSClient
from src.utils.db import get_db

client = GTFSClient()
positions = client.get_vehicle_positions()
print(f'{len(positions)} véhicules actifs récupérés')

inserted = 0
skipped  = 0

with get_db() as db:
    for pos in positions:
        # On ignore les véhicules sur des routes non présentes dans le GTFS statique
        try:
            db.execute(text("""
                INSERT INTO vehicle_positions
                    (vehicle_id, trip_id, route_id, location, bearing, speed, timestamp)
                VALUES (
                    :vid, :tid, :rid,
                    ST_SetSRID(ST_MakePoint(:lon, :lat), 4326),
                    :bearing, :speed, :ts
                )
            """), {
                'vid':     pos.vehicle_id,
                'tid':     pos.trip_id,
                'rid':     pos.route_id,
                'lat':     pos.latitude,
                'lon':     pos.longitude,
                'bearing': pos.bearing,
                'speed':   pos.speed,
                'ts':      pos.timestamp,
            })
            inserted += 1
        except Exception:
            skipped += 1

print(f'✓ {inserted} positions insérées | {skipped} ignorées (route_id inexistante)')

370 véhicules actifs récupérés
✓ 370 positions insérées | 0 ignorées (route_id inexistante)


In [14]:
# Vérifier les données insérées
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM vehicle_positions")).scalar()
    sample = conn.execute(text("""
        SELECT vehicle_id, route_id,
               ST_Y(location) AS lat, ST_X(location) AS lon,
               speed, timestamp
        FROM vehicle_positions
        ORDER BY collected_at DESC
        LIMIT 5
    """)).fetchall()

print(f'Total positions en DB : {count}')
print()
import pandas as pd
pd.DataFrame(sample, columns=['vehicle_id','route_id','lat','lon','speed','timestamp'])

Total positions en DB : 370



,vehicle_id,route_id,lat,lon,speed,timestamp
0,39070,33,45.585045,-73.511070,0.00000,2026-04-02 02:24:09+00:00
1,31115,172,45.469627,-73.535248,0.00000,2026-04-02 02:24:16+00:00
2,40056,18,45.567722,-73.575218,2.50002,2026-04-02 02:24:19+00:00
3,31104,161,45.505642,-73.628189,0.00000,2026-04-02 02:24:17+00:00
4,40065,104,45.471340,-73.670097,7.50006,2026-04-02 02:24:22+00:00


## 4. Activer le collecteur automatique

Les positions sont collectées manuellement ci-dessus. Pour lancer la collecte **automatique toutes les 30 secondes**, ouvre `src/collector/main.py` et décommente les blocs marqués `ÉTAPE 2`.

Ensuite lance le collecteur dans le terminal :

```bash
.venv\Scripts\activate
python -m src.collector.main
```

Ou via Docker Compose (décommenter le service `collector` dans `docker-compose.yml`) :
```bash
docker compose up collector -d
```

In [15]:
# Vérifier la croissance des données après quelques collectes
import time

for i in range(3):
    positions = client.get_vehicle_positions()
    with get_db() as db:
        for pos in positions:
            try:
                db.execute(text("""
                    INSERT INTO vehicle_positions
                        (vehicle_id, trip_id, route_id, location, bearing, speed, timestamp)
                    VALUES (
                        :vid, :tid, :rid,
                        ST_SetSRID(ST_MakePoint(:lon, :lat), 4326),
                        :bearing, :speed, :ts
                    )
                """), {
                    'vid': pos.vehicle_id, 'tid': pos.trip_id, 'rid': pos.route_id,
                    'lat': pos.latitude,   'lon': pos.longitude,
                    'bearing': pos.bearing, 'speed': pos.speed, 'ts': pos.timestamp,
                })
            except Exception:
                pass
    
    with engine.connect() as conn:
        count = conn.execute(text("SELECT COUNT(*) FROM vehicle_positions")).scalar()
    print(f'Collecte {i+1}/3 — {len(positions)} véhicules | Total DB : {count} lignes')
    
    if i < 2:
        time.sleep(30)

Collecte 1/3 — 370 véhicules | Total DB : 740 lignes
Collecte 2/3 — 366 véhicules | Total DB : 1106 lignes
Collecte 3/3 — 365 véhicules | Total DB : 1471 lignes


## 5. Prochaines étapes

- [ ] Activer `src/collector/main.py` (décommenter les blocs ÉTAPE 2)
- [ ] Laisser tourner le collecteur ~1h pour accumuler des données ML
- [ ] Passer au notebook `03_feature_engineering.ipynb` pour construire les features